# 10 – Caching (Redis + In-Memory Fallback)

The cache module provides:
- **Redis** as primary store (with TTL)
- **In-memory dict** as automatic fallback when Redis is unavailable
- **`@cached_node`** decorator for LangGraph nodes
- **`invalidate_pattern`** for cache-busting

All tests use the in-memory fallback — no Redis needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'
os.environ['REDIS_ENABLED'] = 'false'  # force in-memory fallback

In [ ]:
from core.cache import cache_get, cache_set, make_key, invalidate_pattern, _fallback

client = None  # in-memory mode

## 1. Basic set and get

In [ ]:
key = 'test:mykey'
value = {'metrics': {'grr': 92.5}, 'product': 'retention'}

cache_set(client, key, value, ttl=60)
retrieved = cache_get(client, key)

print('Stored   :', value)
print('Retrieved:', retrieved)
print('Match    :', retrieved == value)

## 2. Cache miss

In [ ]:
missing = cache_get(client, 'nonexistent:key')
print('Cache miss returns:', missing)

## 3. TTL expiry

In [ ]:
import time

cache_set(client, 'ttl:test', {'data': 'ephemeral'}, ttl=1)  # 1 second TTL
print('Before expiry:', cache_get(client, 'ttl:test'))

time.sleep(1.1)
print('After expiry :', cache_get(client, 'ttl:test'))  # should be None

## 4. Deterministic key generation

In [ ]:
k1 = make_key('information_agent', query='What is GRR?', data_products=['retention'], time_range='last_30_days')
k2 = make_key('information_agent', query='What is GRR?', data_products=['retention'], time_range='last_30_days')
k3 = make_key('information_agent', query='What is ARR?', data_products=['bookings'], time_range='last_30_days')

print('k1:', k1)
print('k2:', k2)
print('k3:', k3)
print()
print('k1 == k2 (same params):', k1 == k2)
print('k1 == k3 (diff params) :', k1 == k3)

## 5. Pattern invalidation

In [ ]:
# Populate multiple keys
cache_set(client, 'information_agent:abc', {'data': 1}, ttl=300)
cache_set(client, 'information_agent:def', {'data': 2}, ttl=300)
cache_set(client, 'knowledge_agent:xyz', {'data': 3}, ttl=300)

print('Before invalidation:')
print('  information_agent:abc :', cache_get(client, 'information_agent:abc'))
print('  information_agent:def :', cache_get(client, 'information_agent:def'))
print('  knowledge_agent:xyz   :', cache_get(client, 'knowledge_agent:xyz'))

deleted = invalidate_pattern(client, 'information_agent:*')
print(f'\nInvalidated {deleted} key(s) matching information_agent:*')

print('\nAfter invalidation:')
print('  information_agent:abc :', cache_get(client, 'information_agent:abc'))  # gone
print('  information_agent:def :', cache_get(client, 'information_agent:def'))  # gone
print('  knowledge_agent:xyz   :', cache_get(client, 'knowledge_agent:xyz'))    # still there

## 6. cached_node decorator — caches LangGraph node output

In [ ]:
from core.cache import cached_node

call_count = 0

@cached_node('test_node', ttl=60)
def expensive_node(state: dict) -> dict:
    global call_count
    call_count += 1
    return {**state, 'result': f'computed_{call_count}', 'calls': call_count}

state = {'query': 'What is GRR?', 'data_products': ['retention'], 'time_range': 'last_30_days'}

r1 = expensive_node(state)
r2 = expensive_node(state)  # should hit cache
r3 = expensive_node({**state, 'query': 'Different query'})  # cache miss

print(f'Call 1 result  : {r1["result"]}  (calls={r1["calls"]})')
print(f'Call 2 result  : {r2["result"]}  (calls={r2["calls"]})  <- cache HIT')
print(f'Call 3 result  : {r3["result"]}  (calls={r3["calls"]})  <- cache MISS (new query)')
print(f'Total actual fn calls: {call_count}')